In [1]:
import ollama 
import os
from tqdm import tqdm
import json
import signal
import argparse
import wandb
import pandas as pd
import matplotlib.pyplot as plt
import copy
import numpy as np
import re

import sys
from collections import defaultdict

In [2]:
sys.argv = [
    'notebook',  
    '--modelname', 'llama3.2-vision:90b',
    '--data', 'gully',
    '--data_path','/mnt/jacket/WACV-2025-Workshop-ViGIR/results/proposed/test_19Q_llama3.2-vision:90b.json', #test_llama3_90b.json',
    '--subset', 'train',
    '--results_dir', '/mnt/jacket/WACV-2025-Workshop-ViGIR/results/proposed',
    '--timeout', '20',
    '--model_unloading'
]

In [3]:
parser = argparse.ArgumentParser(description="A script to evaluate V-LLMs on different image classification datasets")

parser.add_argument("--modelname", type=str, required=True, help="The name of the V-LLM model")
parser.add_argument("--data", type=str, required=True, help="Dataset name")
parser.add_argument("--data_path", type=str, required=True, help="Path to the image data dir")
parser.add_argument("--subset", type=str, required=True, help="train, test or validation set")
parser.add_argument("--results_dir", type=str, required=True, help="Folder name to save results")
parser.add_argument("--timeout", type=int, default=40, help="time out duration to skip one sample")
parser.add_argument("--model_unloading", action="store_true", help="Enables unloading mode. Every 100 samples it unloades the model from the GPU to avoid carshing.")

args = parser.parse_args()

In [4]:
# Load test set:
file_path = os.path.join(args.data_path)
with open(file_path, 'r') as file:
    data = json.load(file)

print('Number of Annotated GT Images: ', len(data.keys()))
data_keys_test = list(data.keys())
print('Number of Annotated GT Images (List): ', len(data_keys_test))

Number of Annotated GT Images:  311
Number of Annotated GT Images (List):  311


In [5]:
data

{'415': [[{'label': '4', 'labelers': ['Ali', 'Dr.Lory']},
   'Given these six images of the exact same area and collected over a period of 10 years, do you see a low point in the terrain? Answer with yes or no only!',
   'No.'],
  [{'label': '4', 'labelers': ['Ali', 'Dr.Lory']},
   'Given these six images of the exact same area and collected over a period of 10 years, do narrow, winding paths or channels appear? Answer with yes or no only!',
   'No.'],
  [{'label': '4', 'labelers': ['Ali', 'Dr.Lory']},
   'Given these six images of the exact same area and collected over a period of 10 years, are there winding paths that become intermittent recurrent? Answer with yes or no only!',
   'No.'],
  [{'label': '4', 'labelers': ['Ali', 'Dr.Lory']},
   'Given these six images of the exact same area and collected over a period of 10 years, are there any linear depressions or ruts which appear more pronounced along natural drainage lines or slopes? Answer with yes or no only!',
   'No.'],
  [{'la

In [6]:
data_reformated = {}

for key, item in data.items():
    q_and_a = []
    for i in range(len(item)):
        tmp = item[i][1:]   
        q_and_a.append(tmp)

    data_reformated[key] = q_and_a
    

In [7]:
data_reformated

{'415': [['Given these six images of the exact same area and collected over a period of 10 years, do you see a low point in the terrain? Answer with yes or no only!',
   'No.'],
  ['Given these six images of the exact same area and collected over a period of 10 years, do narrow, winding paths or channels appear? Answer with yes or no only!',
   'No.'],
  ['Given these six images of the exact same area and collected over a period of 10 years, are there winding paths that become intermittent recurrent? Answer with yes or no only!',
   'No.'],
  ['Given these six images of the exact same area and collected over a period of 10 years, are there any linear depressions or ruts which appear more pronounced along natural drainage lines or slopes? Answer with yes or no only!',
   'No.'],
  ['Given these six images of the exact same area and collected over a period of 10 years, are there narrow and shallow channels which appear intermittently deeper or more indented into the soil? Answer with yes

In [20]:
num_questions = 12
#idx = [i for i in range(num_questions)]

if num_questions == 3:
    idx = [1,2,13]
elif  num_questions == 6:
    idx = [1,2,13,4,3,5]
elif  num_questions == 9:
    idx = [1,2,13,4,3,5,8,10,12]
elif  num_questions == 12:
    idx = [1,2,13,4,3,5,8,10,12,7,9,6]
elif  num_questions == 15:
    idx = [1,2,13,4,3,5,8,10,12,7,9,6,0,11,14]
elif  num_questions == 18:
    idx = [1,2,13,4,3,5,8,10,12,7,9,6,0,11,14,15,16,17]

data_reformated_final = copy.deepcopy(data_reformated)
for key, item in data_reformated_final.items():
    data_reformated_final[key] = [data_reformated_final[key][i] for i in idx]

In [21]:
data_reformated_final

{'415': [['Given these six images of the exact same area and collected over a period of 10 years, do narrow, winding paths or channels appear? Answer with yes or no only!',
   'No.'],
  ['Given these six images of the exact same area and collected over a period of 10 years, are there winding paths that become intermittent recurrent? Answer with yes or no only!',
   'No.'],
  ['Given these six images of the exact same area and collected over a period of 10 years, are there any branching patterns that resemble temporary streams? Answer with yes or no only!',
   'No.'],
  ['Given these six images of the exact same area and collected over a period of 10 years, are there narrow and shallow channels which appear intermittently deeper or more indented into the soil? Answer with yes or no only!',
   'No.'],
  ['Given these six images of the exact same area and collected over a period of 10 years, are there any linear depressions or ruts which appear more pronounced along natural drainage lines

In [22]:
data_prompt = {}
for key, item in data_reformated_final.items():
    prompt = "\n".join([f"Q: {qa[0]}\nA: {qa[1]}" for qa in item])
    data_prompt[key] = [prompt]

In [23]:
data_prompt

{'415': ['Q: Given these six images of the exact same area and collected over a period of 10 years, do narrow, winding paths or channels appear? Answer with yes or no only!\nA: No.\nQ: Given these six images of the exact same area and collected over a period of 10 years, are there winding paths that become intermittent recurrent? Answer with yes or no only!\nA: No.\nQ: Given these six images of the exact same area and collected over a period of 10 years, are there any branching patterns that resemble temporary streams? Answer with yes or no only!\nA: No.\nQ: Given these six images of the exact same area and collected over a period of 10 years, are there narrow and shallow channels which appear intermittently deeper or more indented into the soil? Answer with yes or no only!\nA: No.\nQ: Given these six images of the exact same area and collected over a period of 10 years, are there any linear depressions or ruts which appear more pronounced along natural drainage lines or slopes? Answer

In [24]:
model_name = args.modelname
ollama.pull(model_name)

timeout_duration = args.timeout

options= {  # new
            "seed": 123,
            "temperature": 0,
            "num_ctx": 2048, # must be set, otherwise slightly random output
        }

model_labels = {}
count = 0

In [25]:
count = 0
saving_response=copy.deepcopy(data_prompt)

for key, info in tqdm(data_prompt.items()):

    #print(key)
    #print(info[0])
    #sys.exit()
    question = "Based on the following questions and their answers, determine if there is evidence of an ephemeral gully in the observed area. Carefully analyze all the questions and the responses to assess.\n\n"
    question += info[0]
    question += "\n\nAfter considering these responses, provide a clear conclusion: Is there evidence of an ephemeral gully? Answer with yes or no only."
    
    #print(question)
    #sys.exit()
        
    count+=1
    #for question in questions:
    response = ollama.generate(model=model_name, 
                               prompt=question, 
                               #images=info, 
                               options=options)
    saving_response[key].append(response['response'])
    print(response['response'])

    #sys.exit()
        

  0%|▍                                                                                                                                       | 1/311 [04:35<23:42:33, 275.33s/it]

Yes.


  1%|▉                                                                                                                                        | 2/311 [04:35<9:44:45, 113.55s/it]

No.


  1%|█▎                                                                                                                                        | 3/311 [04:36<5:18:13, 61.99s/it]

...

No.


100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▌| 310/311 [09:10<00:01,  1.24s/it]

No.
No.


100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 311/311 [09:11<00:00,  1.77s/it]

No.


In [15]:
# Save the dictionary as a JSON file
filename =  'results_'+str(num_questions)+'Q_'+args.modelname+'_raw.json'
with open(os.path.join(args.results_dir, filename), 'w') as json_file:
    json.dump(saving_response, json_file, indent=4)

print(f"JSON file {filename} has been created.")

JSON file results_18Q_llama3.2-vision:90b_raw.json has been created.


In [16]:
def extract_first_yes_no(text):
    # Use regular expressions to find "Yes" or "No" (case-insensitive)
    match = re.search(r'\b(Yes|No)\b', text, re.IGNORECASE)
    if match:
        # Return the first match in its original case
        return match.group(0)
    return None

In [17]:
saving_response_reformat = {}

for key, item in saving_response.items():
    #print(key)
    #print(item[-1])
    text = extract_first_yes_no(item[-1])

    print(f"Image: {key} - Answer: {text}")
    
    if text == 'Yes':
        saving_response_reformat[key] = ["4"]

    if text == 'No':
        saving_response_reformat[key] = ["0"]
        

Image: 415 - Answer: No
Image: 1020 - Answer: No
Image: 105 - Answer: No
Image: 439 - Answer: No
Image: 914 - Answer: Yes
Image: 1099 - Answer: No
Image: 1065 - Answer: No
Image: 373 - Answer: No
Image: 166 - Answer: No
Image: 396 - Answer: No
Image: 837 - Answer: No
Image: 685 - Answer: Yes
Image: 226 - Answer: Yes
Image: 956 - Answer: No
Image: 70 - Answer: No
Image: 604 - Answer: No
Image: 1067 - Answer: No
Image: 118 - Answer: No
Image: 774 - Answer: No
Image: 521 - Answer: No
Image: 910 - Answer: No
Image: 975 - Answer: Yes
Image: 352 - Answer: No
Image: 761 - Answer: No
Image: 1007 - Answer: No
Image: 428 - Answer: No
Image: 1038 - Answer: No
Image: 50 - Answer: No
Image: 838 - Answer: No
Image: 126 - Answer: No
Image: 1078 - Answer: Yes
Image: 944 - Answer: No
Image: 532 - Answer: No
Image: 1041 - Answer: No
Image: 540 - Answer: No
Image: 128 - Answer: No
Image: 722 - Answer: Yes
Image: 478 - Answer: No
Image: 639 - Answer: No
Image: 668 - Answer: Yes
Image: 343 - Answer: No
Ima

In [18]:
saving_response_reformat

{'415': ['0'],
 '1020': ['0'],
 '105': ['0'],
 '439': ['0'],
 '914': ['4'],
 '1099': ['0'],
 '1065': ['0'],
 '373': ['0'],
 '166': ['0'],
 '396': ['0'],
 '837': ['0'],
 '685': ['4'],
 '226': ['4'],
 '956': ['0'],
 '70': ['0'],
 '604': ['0'],
 '1067': ['0'],
 '118': ['0'],
 '774': ['0'],
 '521': ['0'],
 '910': ['0'],
 '975': ['4'],
 '352': ['0'],
 '761': ['0'],
 '1007': ['0'],
 '428': ['0'],
 '1038': ['0'],
 '50': ['0'],
 '838': ['0'],
 '126': ['0'],
 '1078': ['4'],
 '944': ['0'],
 '532': ['0'],
 '1041': ['0'],
 '540': ['0'],
 '128': ['0'],
 '722': ['4'],
 '478': ['0'],
 '639': ['0'],
 '668': ['4'],
 '343': ['0'],
 '1021': ['0'],
 '552': ['0'],
 '848': ['0'],
 '74': ['0'],
 '520': ['0'],
 '1032': ['0'],
 '615': ['0'],
 '536': ['0'],
 '1117': ['0'],
 '646': ['0'],
 '390': ['0'],
 '923': ['0'],
 '194': ['4'],
 '216': ['0'],
 '99': ['0'],
 '372': ['0'],
 '857': ['0'],
 '335': ['4'],
 '505': ['0'],
 '972': ['0'],
 '727': ['0'],
 '1064': ['0'],
 '740': ['4'],
 '182': ['0'],
 '316': ['0'],
 '

In [19]:
# Save the dictionary as a JSON file
filename =  'results_'+str(num_questions)+'Q_'+args.modelname+'_2.json'
with open(os.path.join(args.results_dir, filename), 'w') as json_file:
    json.dump(saving_response_reformat, json_file, indent=4)

print(f"JSON file {filename} has been created.")

JSON file results_18Q_llama3.2-vision:90b.json has been created.
